# Fase 4: Fine-tuning Penerjemah Basa Sunda (loma → lemes keur batur)

**Sebelum mulai**
1. Menu **Runtime → Change runtime type → T4 GPU**.
2. Siapkan file dari komputermu untuk diunggah di langkah 3:
   - `dataset_sunda_loma_lemes_v2.csv` (wajib)
   - `predictions_baseline.csv` dari folder `results/` (opsional, untuk membandingkan dengan baseline kamus)
3. Jalankan sel dari atas ke bawah.

Model default: **ByT5-small** (bekerja per karakter, cocok untuk perubahan ejaan kata seperti *dahar → tuang*). Bisa diganti di langkah 2.

In [ ]:
!pip -q install -U transformers sentencepiece sacrebleu

## 1. Konfigurasi

In [ ]:
import random, re, time
import numpy as np
import pandas as pd
import torch
from sacrebleu.metrics import BLEU, CHRF
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup

# ====== PILIH MODEL (ganti string ini kalau mau eksperimen) ======
MODEL_NAME = "google/byt5-small"
# pilihan lain: "google/mt5-small", "facebook/nllb-200-distilled-600M"

CONFIGS = {
    "google/byt5-small":                dict(lr=3e-4, epochs=40, bs=8, max_src=192, max_new=192),
    "google/mt5-small":                 dict(lr=3e-4, epochs=40, bs=8, max_src=64,  max_new=64),
    "facebook/nllb-200-distilled-600M": dict(lr=5e-5, epochs=15, bs=8, max_src=64,  max_new=64),
}
cfg = CONFIGS[MODEL_NAME]
IS_NLLB = "nllb" in MODEL_NAME
EVAL_EVERY = 2      # evaluasi val tiap N epoch
PATIENCE = 5        # berhenti kalau val tidak membaik setelah N kali evaluasi
SEED = 42

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| model:", MODEL_NAME, "|", cfg)
assert device == "cuda", "Aktifkan GPU: Runtime > Change runtime type > T4 GPU"

## 2. Unggah data

In [ ]:
from google.colab import files
up = files.upload()   # pilih dataset_sunda_loma_lemes_v2.csv (+ predictions_baseline.csv bila ada)

DATA_FILE = next(f for f in up if f.startswith("dataset_sunda"))
BASE_FILE = next((f for f in up if f.startswith("predictions_baseline")), None)

df = pd.read_csv(DATA_FILE)
need = {"id","sumber_loma","target_lemes_batur","sapaan","tipe_kalimat","campuran_sorangan","split"}
assert need <= set(df.columns), f"Kolom kurang: {need - set(df.columns)}"
assert df.isna().sum().sum() == 0
train_df = df[df.split == "train"].reset_index(drop=True)
val_df   = df[df.split == "val"].reset_index(drop=True)
test_df  = df[df.split == "test"].reset_index(drop=True)
print(df.split.value_counts().to_dict(), "| baseline:", BASE_FILE)
df.head(3)

## 3. Muat model dan tokenizer

In [ ]:
if IS_NLLB:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang="sun_Latn", tgt_lang="sun_Latn")
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.0f} juta parameter")

# cek cepat: apakah tokenizer mengenali huruf é dan kata Sunda?
s = "Bapa parantos tuang énjing-énjing?"
ids = tokenizer(text_target=s).input_ids
print(tokenizer.decode(ids, skip_special_tokens=True) == s, "(harus True)")

## 4. Fungsi bantu (batch, terjemah, skor)

In [ ]:
def make_batches(frame, bs, shuffle):
    idx = list(range(len(frame)))
    if shuffle: random.shuffle(idx)
    for i in range(0, len(idx), bs):
        yield frame.iloc[idx[i:i+bs]]

def encode(batch):
    enc = tokenizer(batch.sumber_loma.tolist(), max_length=cfg["max_src"],
                    truncation=True, padding=True, return_tensors="pt")
    lab = tokenizer(text_target=batch.target_lemes_batur.tolist(), max_length=cfg["max_new"],
                    truncation=True, padding=True, return_tensors="pt").input_ids
    lab[lab == tokenizer.pad_token_id] = -100
    out = {k: v.to(device) for k, v in enc.items()}
    out["labels"] = lab.to(device)
    return out

@torch.no_grad()
def translate(sentences, bs=16):
    model.eval()
    kw = dict(num_beams=4, max_new_tokens=cfg["max_new"])
    if IS_NLLB:
        kw["forced_bos_token_id"] = tokenizer.convert_tokens_to_ids("sun_Latn")
    out = []
    for i in range(0, len(sentences), bs):
        enc = tokenizer(sentences[i:i+bs], max_length=cfg["max_src"], truncation=True,
                        padding=True, return_tensors="pt").to(device)
        ids = model.generate(**enc, **kw)
        out += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return [o.strip() for o in out]

def score(refs, hyps):
    return {
        "chrF++": round(CHRF(word_order=2).corpus_score(hyps, [refs]).score, 2),
        "BLEU": round(BLEU().corpus_score(hyps, [refs]).score, 2),
        "exact_match_%": round(100 * sum(h == r for h, r in zip(hyps, refs)) / len(refs), 1),
    }

## 5. Training
Model yang disimpan adalah yang **chrF++ validasinya tertinggi** (data test tidak disentuh selama training).

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=0.01)
steps_per_epoch = -(-len(train_df) // cfg["bs"])
total_steps = steps_per_epoch * cfg["epochs"]
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * total_steps), total_steps)

best, bad, history = -1, 0, []
t0 = time.time()
for epoch in range(1, cfg["epochs"] + 1):
    model.train()
    losses = []
    for batch in make_batches(train_df, cfg["bs"], shuffle=True):
        loss = model(**encode(batch)).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        losses.append(loss.item())
    row = {"epoch": epoch, "train_loss": round(float(np.mean(losses)), 4)}

    if epoch % EVAL_EVERY == 0:
        hyp = translate(val_df.sumber_loma.tolist())
        s = score(val_df.target_lemes_batur.tolist(), hyp)
        row.update({"val_chrF++": s["chrF++"], "val_exact_%": s["exact_match_%"]})
        if s["chrF++"] > best:
            best, bad = s["chrF++"], 0
            model.save_pretrained("best_model"); tokenizer.save_pretrained("best_model")
            row["simpan"] = "*"
        else:
            bad += 1
    history.append(row)
    print(row, f"| {time.time()-t0:.0f}s")
    if bad >= PATIENCE:
        print("Early stopping."); break

pd.DataFrame(history).to_csv("training_log.csv", index=False)
print("val chrF++ terbaik:", best)

## 6. Evaluasi pada data test

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("best_model").to(device)
test_df["pred_model"] = translate(test_df.sumber_loma.tolist())

# tandai kalimat test yang memuat kata yang TIDAK ADA di data train
tok = lambda s: re.findall(r"\w+(?:-\w+)*", s.lower())
train_vocab = {w for s in train_df.sumber_loma for w in tok(s)}
test_df["ada_kata_baru"] = test_df.sumber_loma.apply(lambda s: any(w not in train_vocab for w in tok(s)))

if BASE_FILE:
    b = pd.read_csv(BASE_FILE)[["id", "pred_kamus_v1", "pred_kamus_v2"]]
    test_df = test_df.merge(b, on="id", how="left")

subsets = {
    "test_baru (id>100)": test_df[test_df.id > 100],
    "test_semua": test_df,
    "test: ada kata baru": test_df[test_df.ada_kata_baru],
    "test: tanpa kata baru": test_df[~test_df.ada_kata_baru],
}
systems = [("salin_sumber", "sumber_loma"), ("fine-tune", "pred_model")]
if BASE_FILE:
    systems = [("salin_sumber", "sumber_loma"), ("kamus_v1", "pred_kamus_v1"),
               ("kamus_v2", "pred_kamus_v2"), ("fine-tune", "pred_model")]

rows = []
for sname, sub in subsets.items():
    if len(sub) == 0: continue
    for sys_name, col in systems:
        rows.append({"set": sname, "sistem": sys_name, "n": len(sub),
                     **score(sub.target_lemes_batur.tolist(), sub[col].tolist())})
res = pd.DataFrame(rows)
res.to_csv("hasil_finetune.csv", index=False)
res

In [ ]:
# per kelompok (untuk error analysis di README)
rows = []
for col in ["campuran_sorangan", "tipe_kalimat"]:
    for g, sub in test_df.groupby(col):
        rows.append({"kelompok": f"{col}={g}", "n": len(sub),
                     **score(sub.target_lemes_batur.tolist(), sub.pred_model.tolist())})
pd.DataFrame(rows)

## 7. Contoh kesalahan model

In [ ]:
err = test_df[test_df.pred_model != test_df.target_lemes_batur]
print(f"{len(err)} dari {len(test_df)} kalimat test salah\n")
for r in err.itertuples():
    print(f"[{r.id}] SRC : {r.sumber_loma}\n      REF : {r.target_lemes_batur}\n      PRED: {r.pred_model}\n")

## 8. Simpan hasil

In [ ]:
out_cols = ["id", "sumber_loma", "target_lemes_batur", "pred_model", "ada_kata_baru"]
test_df[out_cols].to_csv("predictions_finetune.csv", index=False)
files.download("predictions_finetune.csv")
files.download("hasil_finetune.csv")
files.download("training_log.csv")

# Model (cukup besar): simpan ke Google Drive agar tidak hilang saat sesi Colab berakhir.
# Hapus tanda # di 3 baris ini kalau mau menyimpan:
# from google.colab import drive; drive.mount("/content/drive")
# import shutil; shutil.copytree("best_model", "/content/drive/MyDrive/sunda_translator_model", dirs_exist_ok=True)
# print("tersimpan di Drive")

## Cara membaca hasil

- Bandingkan **fine-tune** dengan **kamus_v1 / kamus_v2** pada baris `test_baru`, lalu pada `test: ada kata baru`. Bagian kedua adalah ujian sebenarnya: apakah model bisa menangani kata yang tidak ada di train?
- Jika fine-tune kalah dari `kamus_v2`, itu wajar untuk data sekecil ini. Laporkan apa adanya di README.
- Hasil terbaik biasanya muncul setelah menambah data yang lebih beragam. Jalankan ulang notebook dengan dataset baru, ubah hanya `DATA_FILE`.
- Untuk membandingkan model, ganti `MODEL_NAME` di langkah 1 lalu jalankan ulang dari atas. Catat hasil tiap model.